# Train — SmallUNet (Cervical)

omuro データセット（側面・前屈・後屈 合計324件）を使った頚椎8点ランドマーク検出モデルの訓練。

- モデル: SmallUNet
- ランドマーク: C2_center, C2_ant, C2_post, C7_sup_post, C7_inf_ant, C7_inf_post, T1_ant, T1_post
- sigma: 5, loss: AWL, augment: True
- 出力: `SAVE_DIR/best.pt`, `cervical_best.onnx`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content
if [ ! -d spine-measure-assist ]; then
  git clone https://github.com/masaki39/spine-measure-assist.git
fi
cd spine-measure-assist && git pull

In [ ]:
%%bash
pip install uv -q
cd /content/spine-measure-assist
uv sync --extra ml -q

In [ ]:
# ---- 設定（Google Drive内のパスに合わせて変更） ----
DATA_DIR  = "/content/drive/MyDrive/spine_data/omuro_cervical"
SAVE_DIR  = "/content/drive/MyDrive/spine_data/omuro_runs"
LANDMARKS = "C2_center,C2_ant,C2_post,C7_sup_post,C7_inf_ant,C7_inf_post,T1_ant,T1_post"
EPOCHS    = 50
BATCH     = 8

In [ ]:
import subprocess, os
os.makedirs(SAVE_DIR, exist_ok=True)

cmd = [
    "uv", "run", "python", "train/train.py",
    "--data-dir", DATA_DIR,
    "--save-dir", SAVE_DIR,
    "--landmarks", LANDMARKS,
    "--backbone", "smallunet",
    "--sigma", "5",
    "--augment",
    "--loss", "awl",
    "--split-seed", "42",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH),
]
result = subprocess.run(cmd, cwd="/content/spine-measure-assist", capture_output=False)
print("Return code:", result.returncode)

In [ ]:
# ONNX変換
cmd = [
    "uv", "run", "python", "train/export_lumbar.py",
    "--checkpoint", f"{SAVE_DIR}/best.pt",
    "--output",     f"{SAVE_DIR}/cervical_best.onnx",
]
subprocess.run(cmd, cwd="/content/spine-measure-assist")

In [ ]:
# 評価（テストセット）
cmd = [
    "uv", "run", "python", "train/eval_cervical.py",
    "--model",  f"{SAVE_DIR}/cervical_best.onnx",
    "--dir",    DATA_DIR,
    "--splits", f"{SAVE_DIR}/splits.json",
    "--subset", "test",
]
subprocess.run(cmd, cwd="/content/spine-measure-assist")